In [84]:
# =========================================
# CÉLULA 1 — Imports e configurações básicas
# =========================================

from __future__ import annotations

import os
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [85]:
# ==========================================================
# CÉLULA 2 — ETAPA 1: Conectar no Postgres e aplicar o DDL (DW)
# ==========================================================

POSTGRES_DB = os.getenv("POSTGRES_DB", "airline_delay_causes")
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = int(os.getenv("POSTGRES_PORT", "5432"))

DW_SCHEMA = os.getenv("DW_SCHEMA", "dw")  # você pediu dw

connection_string = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)
engine = create_engine(connection_string)

print("=" * 80)
print("ETAPA 1: DDL - Criar estruturas da DW")
print("=" * 80)

ddl_candidates = [
    "../Data Layer/gold/ddl.sql",
    "../Data Layer/dw/ddl.sql",
    "Data Layer/gold/ddl.sql",
    "Data Layer/dw/ddl.sql",
    "ddl.sql",
]

ddl_path = next((p for p in ddl_candidates if os.path.exists(p)), None)
if ddl_path is None:
    raise FileNotFoundError(
        "Não encontrei o ddl.sql. Caminhos testados:\n- " + "\n- ".join(ddl_candidates)
    )

with open(ddl_path, "r", encoding="utf-8") as f:
    ddl_sql = f.read()

# Multi-statements simples (separando por ';')
statements = [s.strip() for s in ddl_sql.split(";") if s.strip()]

with engine.begin() as conn:
    for stmt in statements:
        conn.execute(text(stmt))

print(f"DDL aplicado com sucesso: {ddl_path}")
print(f"Schema alvo esperado: {DW_SCHEMA}")


ETAPA 1: DDL - Criar estruturas da DW
DDL aplicado com sucesso: ../Data Layer/gold/ddl.sql
Schema alvo esperado: dw


In [86]:
# =========================================
# CÉLULA 3 — ETAPA 2: EXTRACT (ler a Silver)
# =========================================

print("=" * 80)
print("ETAPA 2: EXTRACT - Ler dados da SILVER")
print("=" * 80)

df_silver = pd.read_sql_query("SELECT * FROM silver.silver_airline_on_time;", engine)

print("Linhas, colunas (Silver):", df_silver.shape)
display(df_silver.head(5))

print("\nNulos por coluna (top 10):")
display(df_silver.isna().sum().sort_values(ascending=False).head(10))

print("\nDuplicados no grão da Silver (year, month, carrier, airport):")
dup = df_silver.duplicated(subset=["year", "month", "carrier", "airport"]).sum()
print(int(dup))


ETAPA 2: EXTRACT - Ler dados da SILVER


Linhas, colunas (Silver): (279182, 23)


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,is_outlier_arr_delay,flight_date
0,2004,1,DL,Delta Air Lines Inc.,PBI,"West Palm Beach/Palm Beach, FL: Palm Beach Int...",650,126,21,6,52,1,46,4,0,5425.0,881.0,397.0,2016.0,15.0,2116.0,0,2004-01-01
1,2004,1,DL,Delta Air Lines Inc.,PDX,"Portland, OR: Portland International",314,61,14,3,34,0,10,30,3,2801.0,478.0,239.0,1365.0,0.0,719.0,0,2004-01-01
2,2004,1,DL,Delta Air Lines Inc.,PHL,"Philadelphia, PA: Philadelphia International",513,97,28,0,52,0,17,15,0,4261.0,1150.0,16.0,2286.0,0.0,809.0,0,2004-01-01
3,2004,1,DL,Delta Air Lines Inc.,PHX,"Phoenix, AZ: Phoenix Sky Harbor International",334,78,20,2,39,0,16,3,1,3400.0,1159.0,166.0,1295.0,0.0,780.0,0,2004-01-01
4,2004,1,DL,Delta Air Lines Inc.,PIT,"Pittsburgh, PA: Pittsburgh International",217,47,8,0,22,0,17,4,1,1737.0,350.0,28.0,522.0,0.0,837.0,0,2004-01-01



Nulos por coluna (top 10):


year            0
month           0
carrier         0
carrier_name    0
airport         0
airport_name    0
arr_flights     0
arr_del15       0
carrier_ct      0
weather_ct      0
dtype: int64


Duplicados no grão da Silver (year, month, carrier, airport):
0


In [87]:
# ======================================================
# CÉLULA 4 — ETAPA 3: TRANSFORM (criar DataFrames das DIMs)
# ======================================================
# Regra aplicada (simples, correta pro seu escopo):
# - dim_cia e dim_apt: 1 linha por código; se houver conflito de nome, escolhe o MAIS FREQUENTE.
# - dim_tmp: 1 linha por (ano,mês); dat_tmp = 1º dia do mês (âncora).
# - padronização: carrier/airport em MAIÚSCULO e sem espaços.

print("=" * 80)
print("ETAPA 3: TRANSFORM - Montar DIMs (DataFrames)")
print("=" * 80)

df = df_silver.copy()

df["carrier"] = df["carrier"].astype(str).str.strip().str.upper()
df["airport"] = df["airport"].astype(str).str.strip().str.upper()

# ---- DIM_TMP (1 linha por ano/mês) ----
dim_tmp = (
    df[["year", "month"]]
    .drop_duplicates()
    .rename(columns={"year": "num_ano", "month": "num_mes"})
)

dim_tmp["dat_tmp"] = pd.to_datetime(
    dim_tmp["num_ano"].astype(str) + "-" + dim_tmp["num_mes"].astype(str) + "-01",
    errors="coerce"
).dt.date

dim_tmp["num_tri"] = ((dim_tmp["num_mes"] - 1) // 3 + 1).astype("Int64")

mes_pt = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}
dim_tmp["nom_mes"] = dim_tmp["num_mes"].map(mes_pt)

dim_tmp = dim_tmp.sort_values(["num_ano", "num_mes"]).reset_index(drop=True)

print("dim_tmp:", dim_tmp.shape)
display(dim_tmp.head(5))
print("Duplicados dim_tmp (ano,mes):", int(dim_tmp.duplicated(["num_ano", "num_mes"]).sum()))

# ---- DIM_CIA (1 linha por cod_cia; nome MAIS FREQUENTE) ----
tmp_cia = (
    df[["carrier", "carrier_name"]]
    .rename(columns={"carrier": "cod_cia", "carrier_name": "nom_cia"})
    .copy()
)
tmp_cia["nom_cia"] = tmp_cia["nom_cia"].astype(str).str.strip()
tmp_cia.loc[tmp_cia["nom_cia"].isin(["", "None", "nan", "NaN"]), "nom_cia"] = np.nan

cia_freq = (
    tmp_cia
    .dropna(subset=["cod_cia"])
    .groupby(["cod_cia", "nom_cia"], dropna=False)
    .size()
    .reset_index(name="qtd_occ")
)

dim_cia = (
    cia_freq
    .sort_values(["cod_cia", "qtd_occ", "nom_cia"], ascending=[True, False, True])
    .drop_duplicates(subset=["cod_cia"], keep="first")
    .drop(columns=["qtd_occ"])
    .sort_values("cod_cia")
    .reset_index(drop=True)
)

print("\ndim_cia:", dim_cia.shape)
display(dim_cia.head(5))
print("Duplicados dim_cia (cod_cia):", int(dim_cia.duplicated(["cod_cia"]).sum()))

# ---- DIM_APT (1 linha por cod_apt; nome MAIS FREQUENTE) ----
tmp_apt = (
    df[["airport", "airport_name"]]
    .rename(columns={"airport": "cod_apt", "airport_name": "nom_apt"})
    .copy()
)
tmp_apt["nom_apt"] = tmp_apt["nom_apt"].astype(str).str.strip()
tmp_apt.loc[tmp_apt["nom_apt"].isin(["", "None", "nan", "NaN"]), "nom_apt"] = np.nan

apt_freq = (
    tmp_apt
    .dropna(subset=["cod_apt"])
    .groupby(["cod_apt", "nom_apt"], dropna=False)
    .size()
    .reset_index(name="qtd_occ")
)

dim_apt = (
    apt_freq
    .sort_values(["cod_apt", "qtd_occ", "nom_apt"], ascending=[True, False, True])
    .drop_duplicates(subset=["cod_apt"], keep="first")
    .drop(columns=["qtd_occ"])
    .sort_values("cod_apt")
    .reset_index(drop=True)
)

print("\ndim_apt:", dim_apt.shape)
display(dim_apt.head(5))
print("Duplicados dim_apt (cod_apt):", int(dim_apt.duplicated(["cod_apt"]).sum()))


ETAPA 3: TRANSFORM - Montar DIMs (DataFrames)
dim_tmp: (202, 5)


,num_ano,num_mes,dat_tmp,num_tri,nom_mes
0,2003,6,2003-06-01,2,Junho
1,2003,7,2003-07-01,3,Julho
2,2003,8,2003-08-01,3,Agosto
3,2003,9,2003-09-01,3,Setembro
4,2003,10,2003-10-01,4,Outubro


Duplicados dim_tmp (ano,mes): 0

dim_cia: (28, 2)


,cod_cia,nom_cia
0,9E,Pinnacle Airlines Inc.
1,AA,American Airlines Inc.
2,AQ,Aloha Airlines Inc.
3,AS,Alaska Airlines Inc.
4,B6,JetBlue Airways


Duplicados dim_cia (cod_cia): 0

dim_apt: (409, 2)


,cod_apt,nom_apt
0,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ..."
1,ABI,"Abilene, TX: Abilene Regional"
2,ABQ,"Albuquerque, NM: Albuquerque International Sun..."
3,ABR,"Aberdeen, SD: Aberdeen Regional"
4,ABY,"Albany, GA: Southwest Georgia Regional"


Duplicados dim_apt (cod_apt): 0


In [88]:
# ==========================================================
# CÉLULA 5 — ETAPA 4: LOAD (carregar DIMs no banco) [CORRIGIDA]
# ==========================================================
# Correção:
# - TRUNCATE em tabelas referenciadas por FK precisa ser feito:
#   (a) truncando todas no mesmo comando, OU
#   (b) usando TRUNCATE ... CASCADE
# Aqui usamos (a), que é mais controlado.

print("=" * 80)
print("ETAPA 4: LOAD - Inserir DIMs na DW")
print("=" * 80)

# validações mínimas
if dim_tmp[["num_ano", "num_mes", "dat_tmp"]].isna().any().any():
    raise ValueError("DIM_TMP possui nulos em (num_ano, num_mes, dat_tmp).")

if dim_cia[["cod_cia"]].isna().any().any():
    raise ValueError("DIM_CIA possui nulos em cod_cia.")

if dim_apt[["cod_apt"]].isna().any().any():
    raise ValueError("DIM_APT possui nulos em cod_apt.")

# Limpeza: truncar tudo junto para não violar FKs
with engine.begin() as conn:
    conn.execute(text(
        f"TRUNCATE TABLE "
        f"{DW_SCHEMA}.fat_atr, "
        f"{DW_SCHEMA}.dim_tmp, "
        f"{DW_SCHEMA}.dim_cia, "
        f"{DW_SCHEMA}.dim_apt "
        f"RESTART IDENTITY;"
    ))

# Inserções em batch
dim_tmp.to_sql(
    "dim_tmp", con=engine, schema=DW_SCHEMA, if_exists="append",
    index=False, method="multi", chunksize=5000
)
dim_cia.to_sql(
    "dim_cia", con=engine, schema=DW_SCHEMA, if_exists="append",
    index=False, method="multi", chunksize=5000
)
dim_apt.to_sql(
    "dim_apt", con=engine, schema=DW_SCHEMA, if_exists="append",
    index=False, method="multi", chunksize=5000
)

# Conferência
with engine.connect() as conn:
    qtd_tmp = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_tmp;")).scalar()
    qtd_cia = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_cia;")).scalar()
    qtd_apt = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.dim_apt;")).scalar()

print(f"Linhas inseridas: dim_tmp={qtd_tmp}, dim_cia={qtd_cia}, dim_apt={qtd_apt}")


ETAPA 4: LOAD - Inserir DIMs na DW
Linhas inseridas: dim_tmp=202, dim_cia=28, dim_apt=409


In [ ]:
# ==========================================================
# CÉLULA 6 — ETAPA 5: TRANSFORM FINAL (resolver SRKs e montar a FATO)
# ==========================================================
# Observação:
# - Se aparecer SRK faltando, esta célula já imprime exemplos do que não casou.

print("=" * 80)
print("ETAPA 5: TRANSFORM - Resolver SRKs e montar FATO (DataFrame)")
print("=" * 80)

dim_tmp_db = pd.read_sql_query(
    f"SELECT srk_tmp, num_ano, num_mes FROM {DW_SCHEMA}.dim_tmp;",
    engine
)
dim_cia_db = pd.read_sql_query(
    f"SELECT srk_cia, cod_cia FROM {DW_SCHEMA}.dim_cia;",
    engine
)
dim_apt_db = pd.read_sql_query(
    f"SELECT srk_apt, cod_apt FROM {DW_SCHEMA}.dim_apt;",
    engine
)

print("DIMs no banco (linhas):",
      f"dim_tmp={len(dim_tmp_db)}, dim_cia={len(dim_cia_db)}, dim_apt={len(dim_apt_db)}")

if len(dim_tmp_db) == 0 or len(dim_cia_db) == 0 or len(dim_apt_db) == 0:
    raise ValueError("DIM vazia no banco. Rode a CÉLULA 5 antes.")

# padronizar também o lado do banco
dim_cia_db["cod_cia"] = dim_cia_db["cod_cia"].astype(str).str.strip().str.upper()
dim_apt_db["cod_apt"] = dim_apt_db["cod_apt"].astype(str).str.strip().str.upper()
dim_tmp_db["num_ano"] = pd.to_numeric(dim_tmp_db["num_ano"], errors="coerce").astype("Int64")
dim_tmp_db["num_mes"] = pd.to_numeric(dim_tmp_db["num_mes"], errors="coerce").astype("Int64")

df_fact_base = df_silver.copy()
df_fact_base["year"] = pd.to_numeric(df_fact_base["year"], errors="coerce").astype("Int64")
df_fact_base["month"] = pd.to_numeric(df_fact_base["month"], errors="coerce").astype("Int64")
df_fact_base["carrier"] = df_fact_base["carrier"].astype(str).str.strip().str.upper()
df_fact_base["airport"] = df_fact_base["airport"].astype(str).str.strip().str.upper()
df_fact_base = df_fact_base.replace([np.inf, -np.inf], np.nan)

# merges para SRK
df_fact_base = df_fact_base.merge(
    dim_tmp_db, left_on=["year", "month"], right_on=["num_ano", "num_mes"], how="left"
).merge(
    dim_cia_db, left_on=["carrier"], right_on=["cod_cia"], how="left"
).merge(
    dim_apt_db, left_on=["airport"], right_on=["cod_apt"], how="left"
)

missing_tmp = int(df_fact_base["srk_tmp"].isna().sum())
missing_cia = int(df_fact_base["srk_cia"].isna().sum())
missing_apt = int(df_fact_base["srk_apt"].isna().sum())

if missing_tmp or missing_cia or missing_apt:
    msg = [
        "Falha ao resolver SRKs:",
        f"- srk_tmp faltando: {missing_tmp}",
        f"- srk_cia faltando: {missing_cia}",
        f"- srk_apt faltando: {missing_apt}",
    ]
    if missing_tmp:
        ex = df_fact_base.loc[df_fact_base["srk_tmp"].isna(), ["year", "month"]].drop_duplicates().head(10)
        msg.append("\nExemplos year/month sem match:\n" + ex.to_string(index=False))
    if missing_cia:
        ex = df_fact_base.loc[df_fact_base["srk_cia"].isna(), ["carrier"]].drop_duplicates().head(10)
        msg.append("\nExemplos carrier sem match:\n" + ex.to_string(index=False))
    if missing_apt:
        ex = df_fact_base.loc[df_fact_base["srk_apt"].isna(), ["airport"]].drop_duplicates().head(10)
        msg.append("\nExemplos airport sem match:\n" + ex.to_string(index=False))
    raise ValueError("\n".join(msg))

# IMPORTANTE:
# As colunas abaixo assumem o seu DDL “3 letras”:
# qtd_atr_del, val_atr_mnt, qtd_atr_aer_tar, val_tax_atr, val_atr_pvo, etc.
df_fat_atr = pd.DataFrame({
    "srk_tmp": df_fact_base["srk_tmp"].astype("int64"),
    "srk_cia": df_fact_base["srk_cia"].astype("int64"),
    "srk_apt": df_fact_base["srk_apt"].astype("int64"),

    "qtd_voo_arr": pd.to_numeric(df_fact_base["arr_flights"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_del": pd.to_numeric(df_fact_base["arr_del15"], errors="coerce").fillna(0).astype("int64"),
    "qtd_voo_can": pd.to_numeric(df_fact_base["arr_cancelled"], errors="coerce").fillna(0).astype("int64"),
    "qtd_voo_div": pd.to_numeric(df_fact_base["arr_diverted"], errors="coerce").fillna(0).astype("int64"),
    "val_atr_mnt": pd.to_numeric(df_fact_base["arr_delay"], errors="coerce").fillna(0).round(2),

    "qtd_atr_cia":     pd.to_numeric(df_fact_base["carrier_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_cli":     pd.to_numeric(df_fact_base["weather_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_nas":     pd.to_numeric(df_fact_base["nas_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_seg":     pd.to_numeric(df_fact_base["security_ct"], errors="coerce").fillna(0).astype("int64"),
    "qtd_atr_aer_tar": pd.to_numeric(df_fact_base["late_aircraft_ct"], errors="coerce").fillna(0).astype("int64"),

    "val_atr_cia_mnt":     pd.to_numeric(df_fact_base["carrier_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_cli_mnt":     pd.to_numeric(df_fact_base["weather_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_nas_mnt":     pd.to_numeric(df_fact_base["nas_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_seg_mnt":     pd.to_numeric(df_fact_base["security_delay"], errors="coerce").fillna(0).round(2),
    "val_atr_aer_tar_mnt": pd.to_numeric(df_fact_base["late_aircraft_delay"], errors="coerce").fillna(0).round(2),

    "ind_out_atr": df_fact_base["is_outlier_arr_delay"].fillna(False).astype(bool),
})

# KPIs
voos = df_fat_atr["qtd_voo_arr"].astype("float64")
df_fat_atr["val_tax_atr"] = np.where(voos > 0, df_fat_atr["qtd_atr_del"].astype("float64") / voos, np.nan)
df_fat_atr["val_atr_pvo"] = np.where(voos > 0, df_fat_atr["val_atr_mnt"].astype("float64") / voos, np.nan)

# dedup no grão
before = len(df_fat_atr)
df_fat_atr = df_fat_atr.drop_duplicates(subset=["srk_tmp", "srk_cia", "srk_apt"]).reset_index(drop=True)
after = len(df_fat_atr)

print(f"FATO: linhas antes={before}, depois dedup={after}")
display(df_fat_atr.head(5))


ETAPA 5: TRANSFORM - Resolver SRKs e montar FATO (DataFrame)
DIMs no banco (linhas): dim_tmp=202, dim_cia=28, dim_apt=409
FATO: linhas antes=279182, depois dedup=279182


,srk_tmp,srk_cia,srk_apt,qtd_voo_arr,qtd_atr_del,qtd_voo_can,qtd_voo_div,val_atr_mnt,qtd_atr_cia,qtd_atr_cli,qtd_atr_nas,qtd_atr_seg,qtd_atr_aer_tar,val_atr_cia_mnt,val_atr_cli_mnt,val_atr_nas_mnt,val_atr_seg_mnt,val_atr_aer_tar_mnt,ind_out_atr,val_tax_atr,val_atr_pvo
0,8,8,289,650,126,4,0,5425.0,21,6,52,1,46,881.0,397.0,2016.0,15.0,2116.0,False,0.193846,8.346154
1,8,8,290,314,61,30,3,2801.0,14,3,34,0,10,478.0,239.0,1365.0,0.0,719.0,False,0.194268,8.920382
2,8,8,295,513,97,15,0,4261.0,28,0,52,0,17,1150.0,16.0,2286.0,0.0,809.0,False,0.189084,8.306043
3,8,8,296,334,78,3,1,3400.0,20,2,39,0,16,1159.0,166.0,1295.0,0.0,780.0,False,0.233533,10.179641
4,8,8,302,217,47,4,1,1737.0,8,0,22,0,17,350.0,28.0,522.0,0.0,837.0,False,0.216590,8.004608


In [90]:
# ==========================================================
# CÉLULA 7 — ETAPA 6: LOAD (carregar a FATO na DW)
# ==========================================================

print("=" * 80)
print("ETAPA 6: LOAD - Inserir FATO na DW")
print("=" * 80)

with engine.begin() as conn:
    conn.execute(text(f"TRUNCATE TABLE {DW_SCHEMA}.fat_atr RESTART IDENTITY;"))

fat_cols = [
    "srk_tmp", "srk_cia", "srk_apt",
    "qtd_voo_arr", "qtd_atr_del", "qtd_voo_can", "qtd_voo_div", "val_atr_mnt",
    "qtd_atr_cia", "qtd_atr_cli", "qtd_atr_nas", "qtd_atr_seg", "qtd_atr_aer_tar",
    "val_atr_cia_mnt", "val_atr_cli_mnt", "val_atr_nas_mnt", "val_atr_seg_mnt", "val_atr_aer_tar_mnt",
    "ind_out_atr",
    "val_tax_atr", "val_atr_pvo",
]

df_fat_load = df_fat_atr[fat_cols].copy()

df_fat_load.to_sql(
    "fat_atr",
    con=engine,
    schema=DW_SCHEMA,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

with engine.connect() as conn:
    qtd_fat = conn.execute(text(f"SELECT COUNT(*) FROM {DW_SCHEMA}.fat_atr;")).scalar()

print(f"Linhas inseridas na FATO ({DW_SCHEMA}.fat_atr): {qtd_fat}")
print("Linhas no DataFrame da FATO:", len(df_fat_load))


ETAPA 6: LOAD - Inserir FATO na DW
Linhas inseridas na FATO (dw.fat_atr): 279182
Linhas no DataFrame da FATO: 279182
